## **Pyspark vs SQL**

Regla mental que debes repetir en clase

* SQL = declarativo
* PySpark = transformaciones encadenadas sobre Dataframes

In [0]:
%sql
SELECT  * FROM samples.bakehouse.sales_customers
LIMIT 10


In [0]:
df = spark.read.table("samples.bakehouse.sales_customers")
display(df.limit(5))

df_selected = df.select("*")
df_selected.limit(5).display()

## DISTINCT

In [0]:
%sql
SELECT DISTINCT country FROM samples.bakehouse.sales_customers

In [0]:
df_distinct = df.select("state","country","continent").distinct()
display(df_distinct)

## WHERE /FILTER

In [0]:
%sql
SELECT * FROM samples.bakehouse.sales_customers
WHERE 1=1
AND country="Japan"
AND gender="female"
LIMIT 5;

In [0]:
from pyspark.sql.functions import col

df_filtrado = df.filter(
    (col("country") == "Japan") & (col("gender") == "female")
    )
 
display(df_filtrado)

In [0]:
df_filtrado = df.where((df["country"] == "Japan") & (df["gender"] == "female"))
display(df_filtrado)

## ORDER BY + LIMIT

In [0]:
%sql
SELECT * FROM samples.bakehouse.sales_customers
ORDER BY first_name ASC LIMIT 15;

In [0]:
df_ordernado =df.orderBy("first_name",ascending=True)
display(df_ordernado.limit(15))

df_sorted = df.orderBy(col("first_name").desc())
display(df_sorted.limit(15))

## CREAR COLUMNAS

In [0]:
%sql
SELECT 
concat(first_name," ", last_name) as full_name,
concat_ws(", ",city,state,country,continent) as address ,
35 as age,
"random comment" as comment,
true as instrue
FROM samples.bakehouse.sales_customers
LIMIT 10

In [0]:
from pyspark.sql.functions import concat,lit,col
df_plus_columns = df.select(
    concat(col("first_name"),lit(" "),col("last_name")).alias("full_name"),
    lit(True).alias("is_true"),
    lit("random comment").alias("comment")

).limit(10)
display(df_plus_columns)

## GROUP BY + AGGREGATION

In [0]:
%sql
SELECT country,COUNT(*) AS CANTIDAD
FROM samples.bakehouse.sales_franchises
GROUP BY country
order by cantidad desc

In [0]:
from pyspark.sql.functions import count
df = spark.read.table("samples.bakehouse.sales_franchises")

df_agrupado = (
        df.groupBy("country")
        .agg(count("*").alias("cantidad"))
        .orderBy(col("cantidad").desc())

)
display(df_agrupado)

## JOIN (clave para silver)

In [0]:
%sql
SELECT 
concat(c.first_name," ",c.last_name) as full_name,
c.country,
c.gender,
s.product,
s.quantity,
s.paymentMethod,
s.dateTime
from samples.bakehouse.sales_transactions s
join samples.bakehouse.sales_customers c 
on s.customerID=c.customerID

In [0]:
df_transactions = spark.read.table("samples.bakehouse.sales_transactions")
df = spark.read.table("samples.bakehouse.sales_customers")


df_joined = (df.alias("c")
             .join(df_transactions.alias("s"),on="customerID", how="left")
             .select(
                    concat(col("c.first_name"),lit(" "),col("c.last_name")).alias("full_name"),
                    "c.country",    
                    "c.gender",
                    "s.product",
                    "s.quantity",
                    "s.paymentMethod",
                    "s.dateTime"
                     
            )).filter(col("s.product").isNotNull())

display(df_joined)


## CASTEO

In [0]:
%sql
SELECT
CAST(transactionID AS STRING) AS transactionID,
CAST(customerID AS STRING) AS customerID,
CAST(franchiseID AS STRING) AS franchiseID,
CAST(datetime AS STRING) AS datetime,
CAST(product AS STRING) AS product
FROM samples.bakehouse.sales_transactions

In [0]:
df_casted = df_transactions.select(
    col("transactionID").cast("string"),
    col("customerID").cast("string"),
    col("franchiseID").cast("string"),
    col("dateTime").cast("string"),
    col("product").cast("string"),
    (col("unitPrice") * col("quantity")).cast("bigint").alias("totalPrice")
)
display(df_casted)


In [0]:
df_test = df_casted.withColumn("totalPrice2",col("totalPrice"))
df_test = df_casted.withColumn("Nuevacolumnita",lit("hola"))
df_new = df_test.drop("nuevaColumnita")
df_new.display()